Import Libraries and Load Dataset

In [6]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = pd.read_csv("FakeNewsNet.csv")

df.head()

ModuleNotFoundError: No module named 'gensim'

prepare text

In [ ]:
df = df.dropna(subset=['title', 'real'])

df['title'] = df['title'].astype(str)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
    ]
    
    return words

df['tokens'] = df['title'].apply(preprocess_text)

display(df[['title', 'tokens']].head())

Train Word2Vec Model

In [ ]:
sentences = df['tokens'].tolist()

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

print("Word2Vec Model Trained Successfully")
print("Vocabulary Size:", len(model.wv))

Extract Word Embeddings

In [ ]:
word_vectors = model.wv

embedding_data = []

for word in word_vectors.index_to_key:
    embedding_data.append({
        'Word': word,
        'Embedding': word_vectors[word]
    })

embedding_df = pd.DataFrame(embedding_data)

display(embedding_df.head())

Find Similar Words

In [ ]:
word = model.wv.index_to_key[0]

similar_words = model.wv.most_similar(
    word,
    topn=10
)

print("Selected Word:", word)
print("\nSemantically Similar Words:")

for similar_word, score in similar_words:
    print(similar_word, round(score, 4))

In [ ]:
word = "president"

if word in model.wv:
    print(model.wv.most_similar(word, topn=10))
else:
    print("Word not found in vocabulary")

Compare Word Frequencies in Fake and Real News

In [ ]:
fake_sentences = df[df['real'] == 0]['tokens']
real_sentences = df[df['real'] == 1]['tokens']

fake_words = pd.Series([
    word for sentence in fake_sentences
    for word in sentence
])

real_words = pd.Series([
    word for sentence in real_sentences
    for word in sentence
])

print("Most Common Words in Fake News:")
display(fake_words.value_counts().head(15))

print("Most Common Words in Real News:")
display(real_words.value_counts().head(15))

Visualize Word Embeddings Using PCA

In [ ]:
words = model.wv.index_to_key[:30]

vectors = np.array([
    model.wv[word] for word in words
])

pca = PCA(n_components=2)

reduced_vectors = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))

plt.scatter(
    reduced_vectors[:, 0],
    reduced_vectors[:, 1]
)

for i, word in enumerate(words):
    plt.annotate(
        word,
        (reduced_vectors[i, 0], reduced_vectors[i, 1])
    )

plt.title("Word2Vec Word Embeddings using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.show()

Save Trained Word2Vec Model

In [ ]:
model.save("fakenews_word2vec.model")

print("Word2Vec Model Saved Successfully")

final results

In [ ]:
print("WORD2VEC ANALYSIS")

print("\nTotal Articles:", len(df))

print("Vocabulary Size:", len(model.wv))

print("Embedding Dimensions:", model.vector_size)

print("\nSample Word Embeddings:")

for word in model.wv.index_to_key[:10]:
    print(word, model.wv[word][:5])

print("\nWord Similarity Analysis:")

word = model.wv.index_to_key[0]

display(
    pd.DataFrame(
        model.wv.most_similar(word, topn=5),
        columns=['Similar_Word', 'Similarity_Score']
    )
)

WORD2VEC ANALYSIS


NameError: name 'df' is not defined